# Notebook 01: Data Quality & Schema Inspection
## SkillGraph AI - Career Skill Intelligence Platform

This notebook performs data quality profiling, missing value assessment, duplicate detection, and schema validation across raw job postings and occupational taxonomies.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

# Set plotting styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', None)
print("Libraries loaded successfully.")

### 1. Load Dataset Manifest & Raw Data Samples

In [ ]:
manifest_path = Path("../data/dataset_manifest.json")
if manifest_path.exists():
    manifest_df = pd.read_json(manifest_path)
    print("Dataset Manifest Summary:")
    display(manifest_df)
else:
    print("Manifest located at data/dataset_manifest.json")

### 2. Inspect Ingested Job Postings Quality

In [ ]:
# Connect to SQLite database
conn = sqlite3.connect("../data/skillgraph.db")

jobs_df = pd.read_sql_query(
    "SELECT id, title, company, location, employment_type, posted_date, industry FROM job_postings LIMIT 500",
    conn
)
print(f"Sampled {len(jobs_df)} postings from database.")
display(jobs_df.head())

### 3. Missing Value Analysis

In [ ]:
missing = jobs_df.isnull().sum()
missing_pct = (missing / len(jobs_df)) * 100
missing_df = pd.DataFrame({"Missing Count": missing, "Percentage (%)": missing_pct})
display(missing_df[missing_df["Missing Count"] > 0])

fig, ax = plt.subplots(figsize=(8, 4))
missing_pct.plot(kind='bar', color='#0d9488', ax=ax)
ax.set_title("Missing Value Distribution (%) across Job Postings Attributes")
ax.set_ylabel("Missing Percentage (%)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 4. Deduplication & Integrity Metrics

In [ ]:
# Inspect data quality report generated by ETL pipeline
import json
report_path = Path("../data/reports/data_quality_report.json")
if report_path.exists():
    with open(report_path) as f:
        dq_report = json.load(f)
    print("Data Quality Report Summary:")
    print(f"Total Raw Records: {dq_report.get('total_records')}")
    print(f"Duplicates Removed: {dq_report.get('duplicates')} ({dq_report.get('duplicate_rate')}%)")
    print(f"Completeness Score: {dq_report.get('completeness')}%")
    print(f"Date Coverage: {dq_report.get('date_coverage', {}).get('start')} to {dq_report.get('date_coverage', {}).get('end')}")
conn.close()